# v38 — T12 на полном train, один запуск
**Run All**: проверки → один full-train refit → фиксированный final checkpoint → сравнение с v25 → отчёт.
925 train-ID; 1800 шагов; обычный batch 16 (P=8, K=2). Инициализация — исходный pretrained DeiT, не старый T12.
Смесь 90/10, граф 20/3/.50 и candidate-порог зафиксированы. Нового перебора нет; MVP не переключается.
Прежний inner holdout теперь входит в train. Original calibration/validation остаются снаружи; это уже наблюдавшиеся development-наборы.
Код/веса/данные v25–v37 не меняются. Ограничения времени нет.

In [1]:
import os
os.environ['ORT_DISABLE_TELEMETRY'] = '1'  # до runtime-импортов, защита от прежнего SIGABRT

from pathlib import Path
import sys, subprocess, time

DEVICE = 'mps'
RUN_NAME = 'refit_v1'  # то же имя = проверенное продолжение
ROOT = next((p for root in [Path.cwd(), *Path.cwd().parents]
             for p in (root, root / 'Car-classification-MSK')
             if (p / 'training/transreid_full_train.py').is_file()), None)
if ROOT is None:
    raise RuntimeError('Открой notebook внутри Car-classification-MSK')
os.chdir(ROOT)
if str(ROOT) not in sys.path: sys.path.insert(0, str(ROOT))
STARTED = time.perf_counter()
print('Python:', sys.executable)
print('Project:', ROOT)
print('Device:', DEVICE, '| Run:', RUN_NAME, '| time limit: NONE')


Python: /Users/elvsevolod/Desktop/учеба/Учёба 4 курс/Хакатон мск сентябрь/Car-classification-MSK/.venv/bin/python
Project: /Users/elvsevolod/Desktop/учеба/Учёба 4 курс/Хакатон мск сентябрь/Car-classification-MSK
Device: mps | Run: refit_v1 | time limit: NONE


In [2]:
print('Этап 1/4: проверки компонентов и продолжения обучения')
subprocess.run([sys.executable, '-m', 'pytest', '-q',
    'tests/test_transreid_full_train.py', 'tests/test_transreid_system.py',
    'tests/test_dual_role_inference.py', 'tests/test_retrieval_policy.py'],
    check=True, cwd=ROOT, env={**os.environ, 'ORT_DISABLE_TELEMETRY': '1'})
from training import transreid_full_train as experiment


Этап 1/4: проверки компонентов и продолжения обучения
........................................................................ [ 76%]
......................                                                   [100%]
94 passed in 7.51s


In [3]:
print('Этап 2/4: проверка исходных файлов и разделения train/outer; хеширование может занять время')
context = experiment.prepare(run_name=RUN_NAME, device=DEVICE)
print('Output:', context['output'])
print('Signature:', context['signature'])
print('Train identities:', len(context['manifest']['train_ids']))
print('Train images:', len(context['target']))
print('Recipe:', context['trials'][0])
print('Training: 1800 steps; batch 16; logs every 25; resume checkpoint every 50')
print('After training: final step 1800 only, 90% v25 + 10% Full T12')
print('Frozen candidate threshold:', context['manifest']['threshold'])


Этап 2/4: проверка исходных файлов и разделения train/outer; хеширование может занять время
PREFLIGHT: verify saved v36/v37, v25, data and original protocols
PREFLIGHT OK: 925 train-ID, 5717 images, 1800 updates, batch=8x2=16; original calibration/validation excluded from training
Output: /Users/elvsevolod/Desktop/учеба/Учёба 4 курс/Хакатон мск сентябрь/Car-classification-MSK/OSNet-AIN-x1.0/variant_38_transreid_full_train/runs/refit_v1
Signature: 344249c0d98dff205e3ead3b3ade09bbfc75960a331adc77f0f4bb3a579efc49
Train identities: 925
Train images: 5717
Recipe: {'id': 'T12_global_supcon', 'architecture': 'global', 'encoder_lr': 0.0001, 'weight_decay': 0.01, 'metric_loss': 'supcon'}
Training: 1800 steps; batch 16; logs every 25; resume checkpoint every 50
After training: final step 1800 only, 90% v25 + 10% Full T12
Frozen candidate threshold: 0.534365177154541


In [4]:
print('Этап 3/4: одноразовый device smoke → обучение → фиксация → сравнение/экспорт')
results = experiment.run(context, allow_outer=True)
assert results['status'] == 'complete' and results['candidate_unchanged']
assert results['optimizer_updates'] == 1800
assert results['inference_bn_updates'] == 0
assert not results['promoted'] and not results['threshold_fit'] and not results['inner_evaluation']


Этап 3/4: одноразовый device smoke → обучение → фиксация → сравнение/экспорт
CHECK: 13285 protected files and source; no historical edits
STAGE runtime_smoke: start
using stride: 16, and patch number is num_y16 * num_x16
using drop_out rate is : 0.0
using attn_drop_out rate is : 0.0
using drop_path rate is : 0.1
Resized position embedding from size:torch.Size([1, 197, 384]) to size: torch.Size([1, 257, 384]) with height:16 width: 16
DEVICE SMOKE global/supcon: batch 16, loss=9.3156, 1.00s
STAGE runtime_smoke: complete in 1.7s
using stride: 16, and patch number is num_y16 * num_x16
using drop_out rate is : 0.0
using attn_drop_out rate is : 0.0
using drop_path rate is : 0.1
Resized position embedding from size:torch.Size([1, 197, 384]) to size: torch.Size([1, 257, 384]) with height:16 width: 16
TRAIN T12_global_supcon | 25/1800 (horizon 1800) | loss=8.7752 | 00:00:11
TRAIN T12_global_supcon | 50/1800 (horizon 1800) | loss=8.5947 | 00:00:21
TRAIN T12_global_supcon | 75/1800 (horizon 1800)

In [5]:
from IPython.display import Markdown, display
print('Этап 4/4: итоговый отчёт')
display(Markdown((context['output'] / 'REPORT.md').read_text(encoding='utf-8')))
for name, report in results['evaluations']['validation'].items():
    print(name, '| validation mAP@10:', report['ranking']['mAP@10'])
print(f'Текущий запуск notebook: {(time.perf_counter()-STARTED)/60:.1f} мин; не benchmark')
print('Final checkpoint:', results['frozen_candidate']['checkpoint']['path'])
print('Report:', context['output'] / 'REPORT.md')
print('Готово. Рабочий v25 не изменён. Можно прислать: обучение завершилось.')


Этап 4/4: итоговый отчёт


# v38 — один T12 на полном train

Обучение: 925 ID, 5717 изображений, 1800 optimizer updates. Инициализация — исходный официальный DeiT, не старый T12.
Смесь 90% v25 + 10% Full T12 и final checkpoint зафиксированы до обучения; подбора по внешним метрикам нет.

| Split | Система | Raw mAP@10 | Graph mAP@10 | Δ graph к v25, п.п. | Rank-1 |
|---|---|---:|---:|---:|---:|
| calibration | V25_control | 0.800272 | 0.826410 | +0.0000 | 0.841463 |
| calibration | Full_T12_w10 | 0.807573 | 0.829879 | +0.3469 | 0.837398 |
| calibration | T12 740 ID — сохранённый v37, не новый замер | — | 0.834048 | +0.7638 | 0.845528 |
| validation | V25_control | 0.808091 | 0.828957 | +0.0000 | 0.829960 |
| validation | Full_T12_w10 | 0.808594 | 0.825002 | -0.3955 | 0.821862 |
| validation | T12 740 ID — сохранённый v37, не новый замер | — | 0.821831 | -0.7127 | 0.809717 |

Validation AP: улучшилось 15, ухудшилось 22, без изменения 210.
Top-1 исправлен у 2, испорчен у 4; top-10 изменился у 279 запросов.

## Ограничения и воспроизведение

Рецепт T12 сохранён: global384, CE + SupCon, lr=1e-4, wd=.01, seed=20260915, P=8/K=2, 1800 шагов.
Сохранено число шагов, не число проходов по каждому ID: при 925 вместо 740 средняя экспозиция на ID ниже.
Это полный refit с новой 925-классовой головой; 740-ID checkpoint не перезаписан и не используется как initializer.
Один дополнительный disposable smoke-update проверяет устройство и отбрасывается; в рабочие 1800 обновлений не входит.
Прежний inner holdout теперь входит в train: метрики на нём как на holdout не вычисляются.
Original calibration/validation исключены из train, не редактировались; обе уже наблюдались раньше, это development, не независимый тест.
Доля .10 и граф legacy 20/3/.50 фиксированы. Raw mAP — диагностика этой же смеси, не новая кандидатская политика.
Кандидат/confidence/порог/отказ — прежний R1 equal3: candidates.csv совпадает с v25 побайтно.
Три файла экспорта: tasks/<split>_<system>/export/. Контроль 2048D, новая система 2432D; реальные unit-блоки с явными slices.
model_provenance.json содержит происхождение новых весов; NPY replay проверяет точные решения. Первая половина системы v25 не переобучается.
Resume сохраняет optimizer/RNG каждые 50 шагов; логи — каждые 25. На прерывании повторяется только несохранённый остаток.
Отработанные этапы проверяются по хешам, кэш не выдаётся за новое обучение или замер скорости. Ограничения времени нет.
Проверены batch 1/8/16/32, порядок/удаление query и неизменность inference-буферов; допуски не увеличивались.
Автоматического promotion, ONNX-переноса в продукт, GPU/Linux benchmark и нового порога нет. Рабочий MVP_fusion_v25 остаётся активным.
NiVe, OCR, изменение bbox и дополнительные metadata-входы не используются; junk/top-10 и остаточный номерной сигнал остаются открытыми ограничениями.


V25_control | validation mAP@10: 0.8289573904235558
Full_T12_w10 | validation mAP@10: 0.8250021956600904
Текущий запуск notebook: 15.3 мин; не benchmark
Final checkpoint: /Users/elvsevolod/Desktop/учеба/Учёба 4 курс/Хакатон мск сентябрь/Car-classification-MSK/OSNet-AIN-x1.0/variant_38_transreid_full_train/runs/refit_v1/training/T12_global_supcon/step_01800.pt
Report: /Users/elvsevolod/Desktop/учеба/Учёба 4 курс/Хакатон мск сентябрь/Car-classification-MSK/OSNet-AIN-x1.0/variant_38_transreid_full_train/runs/refit_v1/REPORT.md
Готово. Рабочий v25 не изменён. Можно прислать: обучение завершилось.
